# Pilot 9 批量迁移实验：评估器问题诊断与修复前基线

- 记录日期：2026-09-24
- 实验批次：`pilot_9`
- 模型：`qwen-plus`（provider: `qwen`）
- 最大修复轮数：3
- 本文档性质：修复评估器前的不可变诊断记录，不作为最终效果结论。

本轮原计划依次运行 3 个 simple、3 个 medium、3 个 complex。simple 与 medium 共 6 个样本完成后，发现评估器会把人工确认过的参考 San 也判为失败，因此暂停 complex，避免继续消耗 token 并产生不可比较数据。

## 1. 可追溯性与冻结快照

修复前全部实验产物与实际运行代码已冻结到：

`data/experiments/snapshots/pilot_9_pre_evaluator_fix_20260924_150916/`

快照包含：批次汇总、6 个样本的逐轮修复报告、全部初始/修复 San 文件、批次配置，以及 SSM、验证器、轻量渲染器、修复提示和服务端提示构造代码。完整文件哈希见 `SHA256SUMS.txt`。

- Git HEAD：`a943106cd071bd9cae96e8cbf33715fc6a5824b0`
- 配置 SHA-256：`15d5cae17a802f0bd8b88a96423236d7eb0f967124547c773aee725a22b92115`
- 原始汇总 SHA-256：`8ca6e8c9498d99062361e7f6f9edb16196a8529de2f6ecfd1be8428b2bfb93c0`
- 注意：运行时工作区含未提交的 SSM 与批量 Runner 改动，因此快照额外保存了关键代码，不能只依赖 Git commit 复现。

In [ ]:
from pathlib import Path
import hashlib
import json

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'migration_pipeline').is_dir() and (path / 'data').is_dir()
)
snapshot = project_root / 'data/experiments/snapshots/pilot_9_pre_evaluator_fix_20260924_150916'
summary = json.loads((snapshot / 'batch_reports/pilot_9_summary.json').read_text(encoding='utf-8'))
summary['aggregate']

## 2. 修复前原始实验结果

| 复杂度 | 样本 | 首次通过 | 修复轮数 | 最终标记 | 总 token |
|---|---|---:|---:|---:|---:|
| simple | step | 否 | 1 | 通过 | 14,637 |
| simple | BookmarkToggleCard | 否 | 1 | 通过 | 18,570 |
| simple | StarRatingInput | 否 | 1 | 通过 | 22,865 |
| medium | KanbanColumnBoard | 否 | 1 | 通过 | 36,750 |
| medium | DataPaginationTable | 否 | 3 | 未通过 | 94,911 |
| medium | FormRulePlayground | 否 | 3 | 未通过 | 74,660 |

截至暂停时：完成 6/9，最终标记通过 4/6，总 token 为 262,393。simple 标记通过率 100%，medium 标记通过率 33.3%，但由于评估器缺陷，medium 指标不能直接解释为模型真实迁移成功率。

In [ ]:
rows = []
for item in summary['samples']:
    rows.append({
        'id': item['id'],
        'complexity': item['complexity'],
        'final_passed': item.get('final_passed'),
        'repair_rounds': item.get('repair_rounds'),
        'initial_validation_errors': item.get('initial_validation_error_codes', []),
        'initial_visual_errors': item.get('initial_visual_error_codes', []),
        'final_validation_errors': item.get('final_validation_error_codes', []),
        'final_visual_errors': item.get('final_visual_error_codes', []),
        'total_tokens': item.get('total_tokens'),
    })
rows

## 3. 为什么判定存在评估器误判

把数据集中的人工确认参考 San 作为正对照输入同一评估器：

| 正对照 | 静态校验 | 视觉结构校验 | 主要报告 |
|---|---:|---:|---|
| DataPaginationTable 参考 San | 失败 | 失败 | `missing_event_bindings`、computed 执行错误、结构相似度 0.1286 |
| FormRulePlayground 参考 San | 失败 | 失败 | `missing_event_bindings`、模板 methods 不可见、结构相似度 0.6111 |

正对照不应失败。因此，当前失败集合同时包含真实迁移缺陷和测量工具缺陷，不能直接用于比较不同复杂度的迁移能力。

## 4. 已定位的评估器缺陷

1. **事件完整性正则误判带引号参数的 handler**：`setSort('id')`、`updateField('name', $event)` 明明存在于 `on-*`，但旧正则禁止属性表达式内部出现引号。
2. **Vue 轻量渲染器没有把 methods 加入模板表达式作用域**：`isEnabled(rule.id, enabled)` 被报告为未定义。
3. **San 轻量渲染器没有把已计算 computed 值写回 data store**：后续 computed 通过 `this.data.get('sortedRows')` 读取依赖时得到 `undefined`。
4. **Vue/San 条件节点处理不一致**：Vue `v-if=false` 删除节点，而 San `s-if=false` 仅增加隐藏标记，导致节点数量和树编辑距离不可比较。
5. **San 模板方法表达式支持不足**：参考模板中的 `isEnabled(...)` 无法稳定参与属性求值。

## 5. 与误判并存的真实迁移问题

评估器存在问题不代表模型输出正确。人工检查确认：

- **DataPaginationTable**：生成代码使用 `this.computed.sortedRows()` 和 `this.computed.pageCount()`；San 中应通过 `this.data.get('sortedRows')`、`this.data.get('pageCount')` 读取 computed，属于真实运行时缺陷。
- **FormRulePlayground**：最终代码把 `form.age` 从 `18` 改成空字符串，并把 `enabled` 从 `['email', 'adult', 'name']` 改成 `['email', 'adult']`，初始语义发生漂移；错误提示文本也与源组件不一致。
- **跨样本问题**：6 个已运行样本首次生成均出现 `missing_prop_defaults`，说明生成阶段对明确提示的遵循仍不稳定，而修复阶段在 simple/Kanban 上能够恢复。

## 6. 修复与重新实验协议

1. 先增加参考 San 正对照回归测试，再修复验证器和两个轻量渲染器。
2. 验收条件：DataPaginationTable、FormRulePlayground、KanbanColumnBoard 三个参考 San 的静态与视觉评估均通过，且现有回归测试全部通过。
3. 不覆盖本快照；修复后的实验使用新批次 ID `pilot_9_v2` 和独立输出目录。
4. v2 使用相同 9 个样本、相同模型、相同最大修复轮数。若模型或 prompt 改变，必须在 manifest 中单独声明。
5. v2 先 dry-run，再运行 simple/medium；确认评估器稳定后才运行 complex。
6. 论文中将本轮标记为“测量工具校准实验”，不得与 v2 的正式成功率直接合并。

## 7. 评估器修复结果（同日）

已修改：

- `migration_pipeline/stages/validate.py`：解析 `on-*` 属性后提取 handler，不再因字符串参数中的引号误判。
- `migration_pipeline/utils/vue_render.py`：把 Vue methods 暴露给模板表达式。
- `migration_pipeline/utils/san_render.py`：computed 按顺序写回 data store、模板可调用组件方法，并统一 `s-if=false` 的节点移除语义。
- `tests/test_migration_pipeline_regressions.py`：把三个 medium 人工参考 San 固化为正对照。

修复后正对照结果：KanbanColumnBoard、DataPaginationTable、FormRulePlayground 的静态校验、结构相似度、标签序列相似度和文本相似度全部通过，三项相似度均为 1.0；完整回归共 13 项通过。

用修复后的评估器重新评估旧模型产物（未调用模型、未改变旧汇总）：

| 旧模型最终产物 | 静态校验 | 视觉校验 | 结论 |
|---|---:|---:|---|
| KanbanColumnBoard | 通过 | 通过（1.0） | 原通过结论保持 |
| DataPaginationTable | 失败：`vue_this_state` | 失败（0.1714） | `this.computed.*` 等真实缺陷仍被识别 |
| FormRulePlayground | 通过（有提示） | 失败（0.9286） | 初始状态漂移仍造成结构差异 |

这说明修复消除了已知假阳性，同时没有把已确认的模型缺陷放行。

## 8. Pilot 9 v2：Simple/Medium 阶段结论与 Prompt 改进基线

v2 在评估器校准后重新运行了全部 3 个 simple 和 3 个 medium，complex 暂停。结果已冻结到：

`data/experiments/snapshots/pilot_9_v2_pre_prompt_fix_20260924_155149/`

| 复杂度 | 样本 | 首次通过 | 修复轮数 | 最终结果 | Token |
|---|---|---:|---:|---:|---:|
| simple | step | 否 | 1 | 通过 | 14,637 |
| simple | BookmarkToggleCard | 否 | 1 | 通过 | 18,570 |
| simple | StarRatingInput | 是 | 0 | 通过 | 10,568 |
| medium | KanbanColumnBoard | 否 | 1 | 通过 | 36,660 |
| medium | DataPaginationTable | 否 | 3 | 失败 | 94,364 |
| medium | FormRulePlayground | 否 | 3 | 失败 | 72,843 |

累计 247,642 token，最终 4/6 通过。DataPaginationTable 三轮始终保留 `this.computed.sortedRows()` / `this.computed.pageCount()`；现有 repair prompt 没有明确规定 San computed 必须通过 `this.data.get('computedName')` 读取。

FormRulePlayground 第一轮后结构相似度升至 0.9286，但保留了四处属性差异：三个 checkbox 使用不可求值的 `checked="{= isEnabled(...) =}"`，且 `form.age` 从 18 漂移为空字符串；同时 `enabled` 丢失 `name`。第 1、2、3 轮修复文件 SHA-256 完全一致，证明后两轮没有任何代码进展，额外消耗 37,788 token。

下一轮改进应同时处理：computed 访问规则、内部 data 默认值精确保真、派生 checkbox 使用单向插值，以及连续评估签名不变时提前终止修复。v2 不继续运行 complex，避免在已知 repair prompt 缺陷下扩大成本。

In [ ]:
# 校验冻结快照完整性
expected = {}
for line in (snapshot / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines():
    digest, relative = line.split('  ', 1)
    expected[relative] = digest

mismatches = []
for relative, digest in expected.items():
    actual = hashlib.sha256((snapshot / relative).read_bytes()).hexdigest()
    if actual != digest:
        mismatches.append((relative, digest, actual))
assert not mismatches, mismatches

manifest = json.loads((snapshot / 'manifest.json').read_text(encoding='utf-8'))
input_mismatches = []
for relative, digest in manifest['input_artifacts'].items():
    actual = hashlib.sha256((project_root / relative).read_bytes()).hexdigest()
    if actual != digest:
        input_mismatches.append((relative, digest, actual))
assert not input_mismatches, input_mismatches
print(f'Integrity verified: {len(expected)} pre-evaluator-fix files and {len(manifest["input_artifacts"])} inputs')

v2_snapshot = project_root / 'data/experiments/snapshots/pilot_9_v2_pre_prompt_fix_20260924_155149'
v2_expected = {}
for line in (v2_snapshot / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines():
    digest, relative = line.split('  ', 1)
    v2_expected[relative] = digest
v2_mismatches = []
for relative, digest in v2_expected.items():
    actual = hashlib.sha256((v2_snapshot / relative).read_bytes()).hexdigest()
    if actual != digest:
        v2_mismatches.append((relative, digest, actual))
assert not v2_mismatches, v2_mismatches
print(f'Integrity verified: {len(v2_expected)} pre-prompt-fix v2 files')

## 9. Pilot 9 v3：修复策略改进与实验隔离

针对 v2 暴露的问题，已建立独立配置 `migration_pipeline/batch_configs/pilot_9_v3.json`，保持评估器、样本和最大修复轮数不变，仅改变修复策略，以便后续进行可归因的 v2/v3 对比。

Repair Prompt 新增四项硬约束：San computed 必须通过 `this.data.get('computedName')` 读取；禁止 `this.computed.foo()`；Vue `data()` 的嵌套对象、数组、数字和启用集合必须精确保留；`{= ... =}` 仅用于可赋值路径，方法调用或 computed 等派生表达式使用 `{{ ... }}`。

编排器为每轮修复记录 progress signature，包括规范化代码 SHA-256、validation/visual error codes、树编辑距离及 structure/tag/text similarity。连续两轮签名完全一致且仍未通过时，以 `stop_reason=no_repair_progress` 提前结束；两轮记录均保留在 repair history，便于统计节省的轮数与 token。

本地验证结果：15 项 unittest 全部通过，v3 dry-run 解析出 3 simple、3 medium、3 complex，输出与报告均隔离到 `pilot_9_v3` 目录；dry-run 未创建 summary、输出目录或报告目录，也未调用模型。正式模型实验尚未开始。

## 10. Pilot 9 v3：两个针对性样本的正式运行结果

2026-09-24 正式运行 `medium_data_pagination_table` 与 `medium_form_rule_playground`。两者均在一轮 repair 后被 pipeline 标记为通过，未触发 `no_repair_progress`。合计消耗 80,781 token，其中 generation 37,194、repair 43,587。相比 v2 两个样本的 167,207 token，减少 86,426（约 51.7%）。

| 样本 | v2 结果/轮数/Token | v3 标记/轮数/Token | 源码复核结论 |
|---|---|---|---|
| DataPaginationTable | 失败 / 3 / 94,364 | 通过 / 1 / 45,438 | `this.computed.*` 已消除，改为 `this.data.get(...)`；结构、标签、文本相似度均为 1.0 |
| FormRulePlayground | 失败 / 3 / 72,843 | 通过 / 1 / 35,343 | checkbox 已改为单向插值，但 `age` 仍由 18 漂移为 `''`，`enabled` 仍缺少 `name`，属于假通过 |

FormRulePlayground 的最终 DOM 仍有两处属性差异：第三个规则 checkbox 缺少 `checked=true`，年龄输入框 value 从 `18` 变为空字符串。tree edit distance 为 0.5，structure similarity 为 0.9643，高于当前 0.95 阈值，因此 visual evaluator 没有产生错误。

根因不是 Prompt 文案缺失，而是 SSM 中 `script.options.data[*].default_value_summary` 被截断到 50 个字符：`form` 摘要在 `age` 前截断，`enabled` 摘要在 `name` 前截断。Repair Prompt 要求精确保留初始值，但模型收到的 SSM 本身没有完整值；validator 也只检查 data 字段是否存在，没有验证初始值等价性。

结论：v3 验证了 computed 规则有效，并显著减少 repair 成本，但尚未解决内部 data 默认值保真问题。由于 v3 已产生正式结果，后续若修改 SSM、validator 或评估规则，应先冻结本轮产物，再建立 v4，不能覆盖 v3。

## 11. Pilot 9 v3 冻结与 v4 准备

v3 两个针对性样本及其运行代码已冻结到 `data/experiments/snapshots/pilot_9_v3_targeted_pre_ssm_fix_20260924_162405/`。快照包含 21 个产物、报告、配置和关键代码文件，SHA-256 复核差异为 0。manifest 明确记录 pipeline 标记通过 2 个、人工源码复核真实通过 1 个、假通过样本为 `medium_form_rule_playground`。

v4 保持样本、Repair Prompt 主体和无进展终止机制不变，仅修复 v3 新发现的测量与输入信息缺口：

1. SSM 的 `script.options.data[*]` 新增未截断的 `default_value_expression`、结构化 `default_value` 和可靠性标记 `default_value_static`；Node/Babel 与 Python fallback 均覆盖。
2. Repair Prompt 的 expected contract 明确列出每个 data 字段的完整初始表达式和静态值。
3. San 编译检查返回 `initData()` 的结构化结果；Validator 对可静态求值字段执行深度等价比较，错误码为 `data_initial_value_mismatch`。
4. visual evaluator 将 `value`、`checked`、`selected`、`disabled` 的差异升级为硬错误 `dom_critical_attribute_mismatch`，不再允许由总体 0.95 结构阈值掩盖。

验证结果：17 项 unittest 全部通过；v3 FormRulePlayground 旧产物现在会同时被上述两个错误码拒绝；`pilot_9_v4` dry-run 得到 3 simple、3 medium、3 complex，配置指纹为 `631ae0884ad06f2d4b1bcd8eca4ce1f0373158ea870e97acaf9bca0526e826cc`；未创建 v4 summary、输出或报告目录，尚未调用模型。

## 12. Pilot 9 v4：两个针对性样本的正式运行结果

v4 正式重跑两个 medium 针对性样本，均在一轮 repair 后通过，未触发 `no_repair_progress`。合计 82,567 token，其中 generation 37,456、repair 45,111。相较 v3 的 80,781 token 增加 1,786（约 2.2%）。

| 样本 | 修复轮数 | Token | 最终静态校验 | 最终 DOM 指标 | 源码复核 |
|---|---:|---:|---|---|---|
| DataPaginationTable | 1 | 46,880 | 通过 | structure/tag/text = 1.0/1.0/1.0 | computed 继续正确使用 `this.data.get(...)` |
| FormRulePlayground | 1 | 35,687 | 通过（1 个 migration hint 警告） | structure/tag/text = 1.0/1.0/1.0，tree distance = 0 | `age: 18`、完整 enabled 集合和 checkbox 初始选中状态均恢复 |

v4 已解决其目标缺陷：v3 FormRulePlayground 的两个关键属性差异从 tree distance 0.5 降为 0，Validator 的结构化初始值检查也通过。

源码复核同时发现一个不属于 v4 原定修复范围的残余语义漂移：Vue 使用 `邮箱格式无效`、`年龄必须不少于 18` 和 `Number(form.age)`，v4 San 使用 `邮箱格式不正确`、`年龄不少于 18` 和 `parseInt(form.age)`。这些逻辑只在提交后显示/执行，当前初始态轻量 DOM 评估无法覆盖。SSM 的 computed `getter_body` 仍截断为 200 个字符，后半段规则没有完整进入生成输入。

因此 v4 可以视为“初始状态保真修复成功”，但不能视为完整交互语义等价。若继续修复 computed/method 完整源码与交互场景评估，应先冻结 v4，再建立新版本，避免覆盖本轮正式结果。

## 13. Pilot 9 v4 冻结与 v5 交互评估准备

v4 两个针对性样本和运行代码已冻结到 `data/experiments/snapshots/pilot_9_v4_targeted_pre_interaction_fix_20260924_164548/`，包含 24 个哈希文件。v4 解决了 FormRulePlayground 的初始状态漂移，但源码复核发现提交后的错误文本和数值转换仍与 Vue 不完全一致。

v5 新增：
- `getter_body_full`、`body_full`：SSM 不再只向生成器提供 200/500 字符截断片段；
- `interaction_eval.py`：用确定性的内存组件上下文执行配置中的 set/call 步骤，比较 Vue/San 每一步 data、computed 和 fired events；这不是浏览器自动化，而是提交真实浏览器前的快速语义检查；
- `pilot_9_v5.json`：FormRulePlayground 配置 `invalid_submission` 场景，填入无效姓名、邮箱和年龄后调用 submit；场景不一致时批次结果标记为 `interaction_eval_failed`。

本地回归测试共 19 项通过，v5 dry-run 配置为 3 simple、3 medium、3 complex，尚未调用模型。已用 v4 旧产物验证交互评估能够发现错误提示文本和数值转换漂移。

## 14. Pilot 9 v5：正式定向运行、比较器校准与最终结果

v5 正式运行 `medium_form_rule_playground`。初次生成未通过：静态校验报告 `missing_prop_defaults`，初始 structure similarity 为 0.2857；一轮 repair 后，静态校验、初始 DOM 结构、标签序列和文本相似度均达到 1.0，并恢复 `age: 18`、完整 enabled 集合、Vue 原始错误文本和 `Number(form.age)`。本次 generation 消耗 16,680 token，repair 消耗 20,168 token，总计 36,848 token。

首次交互评估把该产物标记为 `interaction_eval_failed`。复核显示业务状态与触发事件已经一致，差异来自 San 执行器为 computed 缓存写入的内部 data 字段；比较器将这些运行时派生字段错误地当成了业务 data，属于评估器假阴性，而非迁移代码失败。该原始结果已冻结到 `data/experiments/snapshots/pilot_9_v5_pre_interaction_comparator_fix_20260924_170536/`，包含 11 个哈希文件，完整性差异为 0。

修复比较器的 computed 缓存归一化后，使用 `--reevaluate-interactions --sample-id medium_form_rule_playground` 只重评现有产物，没有再次调用模型，也没有增加 token。最终结果为：

```text
final_passed: true
stop_reason: passed
interaction_eval_passed: true
interaction_eval_reason: interaction_scenarios_passed
interaction_reevaluated_at: 2026-09-24T17:27:22+08:00
model_called: false
```

校准后的最终结果与关键运行代码冻结到 `data/experiments/snapshots/pilot_9_v5_targeted_final_20260924_172722/`，包含 24 个哈希文件；21 项相关回归测试全部通过。两个 v5 快照必须同时保留：前者证明首次假阴性的形成过程，后者证明修正比较口径后同一模型产物无需重新生成即可通过。

当前证据边界仍然是轻量、确定性的内存交互检查，不是真实浏览器自动化，也不覆盖 CSS 布局、截图像素差异和浏览器运行时边缘行为。下一次模型实验建议先运行 v5 的 `medium_data_pagination_table` 作为同版本对照，确认新增交互层不会破坏已知正确的 computed 场景，再扩大到 3 个 complex 样本。

In [ ]:
# 校验 v5 修复前与校准后两个冻结快照的完整性
def verify_snapshot(snapshot_dir):
    expected = {}
    for line in (snapshot_dir / 'SHA256SUMS.txt').read_text(encoding='utf-8').splitlines():
        digest, relative = line.split('  ', 1)
        expected[relative] = digest
    mismatches = []
    for relative, digest in expected.items():
        actual = hashlib.sha256((snapshot_dir / relative).read_bytes()).hexdigest()
        if actual != digest:
            mismatches.append((relative, digest, actual))
    assert not mismatches, mismatches
    return len(expected)

v5_pre_fix = project_root / 'data/experiments/snapshots/pilot_9_v5_pre_interaction_comparator_fix_20260924_170536'
v5_final = project_root / 'data/experiments/snapshots/pilot_9_v5_targeted_final_20260924_172722'
print('v5 pre-fix files:', verify_snapshot(v5_pre_fix))
print('v5 final files:', verify_snapshot(v5_final))

final_summary = json.loads((v5_final / 'batch_reports/pilot_9_v5_summary.json').read_text(encoding='utf-8'))
sample = final_summary['samples'][0]
assert sample['final_passed'] is True
assert sample['interaction_eval_passed'] is True
assert final_summary['interaction_reevaluation']['model_called'] is False
print('Final v5 targeted result verified without an additional model call.')

## 15. Pilot 9 v5：DataPaginationTable 对照失败与 v6 修复

2026-10-08 运行 `medium_data_pagination_table`。样本成功调用模型并完成 3 轮 repair，但最终 `final_passed=false`、`stop_reason=max_repair_rounds_reached`。generation 消耗 21,446 token，repair 消耗 76,285 token，总计 97,731 token。v5 当前两个 medium 样本合计 134,579 token，通过 1/2。

最终静态校验为 34/34，通过；DOM 标签和文本相似度均为 1.0，结构相似度为 0.9857，tree edit distance 为 0.5。唯一硬错误是 `dom_critical_attribute_mismatch`：模型持续输出 `disabled="{= page === 1 =}"` 和 `disabled="{= page === pageCount =}"`。`{= ... =}` 是双向绑定，只适用于可赋值路径；比较表达式属于派生值，应使用 `disabled="{{ ... }}"` 单向插值。轻量 San renderer 因而把错误表达式当成字面属性，正确暴露了两个分页按钮的禁用状态差异。

repair 轨迹显示：第 1 轮已达到 structure 0.9857；第 2 轮回退到 0.1714 并重新出现 `missing_prop_defaults`；第 3 轮恢复到 0.9857，但绑定错误仍未消除。这说明仅依赖视觉错误反馈不足以稳定引导模型修复具体的 San 绑定语法。

因此建立独立批次 `pilot_9_v6`，不覆盖 v5：

- Validator 新增 `invalid_dynamic_attribute_binding`，在 repair 前直接识别 disabled/checked/selected 上的 `{= ... =}`；
- generation 与 Repair Prompt 都明确这些派生属性必须使用 `{{ ... }}`，并给出禁止/正确示例；
- 合法可赋值路径（例如 `value="{= pageSize =}"`）仍允许双向绑定；
- 新增正反回归测试，旧 v5 最终产物现会被静态校验准确拒绝；
- v6 dry-run 得到 3 simple、3 medium、3 complex，配置指纹为 `0de2c5b627e627b6b58a71bc4ce7bf502bd40cfe54f1b48f4dc2f4f59c61a941`，未调用模型。

下一步只重跑 v6 的 `medium_data_pagination_table`，验证新规则能否在较少 repair 轮数内消除绑定错误；通过后再考虑 complex 样本。

## 16. Pilot 9 v6：DataPaginationTable 针对性重跑结果

2026-10-08 使用相同样本、provider `qwen`、模型 `qwen-plus` 和最多 3 轮 repair 的设置，正式运行 `pilot_9_v6/medium_data_pagination_table`。v6 的实验变量是：generation Prompt 与 Repair Prompt 都明确禁止在 `disabled`、`checked`、`selected` 等派生属性上使用 `{= ... =}`；Validator 新增 `invalid_dynamic_attribute_binding`，使该错误能够在静态阶段被直接识别。

最终结果为 `final_passed=true`、`stop_reason=passed`，仅执行 1 轮 repair。最终静态校验通过，Vue/San 均为 35 个节点，tree edit distance 为 0，structure/tag/text similarity 均为 1.0。生成结果和修复结果都正确使用：

```san
<select value="{= pageSize =}">
<button disabled="{{ page === 1 }}">上一页</button>
<button disabled="{{ page === pageCount }}">下一页</button>
```

这表明可赋值路径仍可使用双向绑定，而 derived disabled 状态已经稳定迁移为单向插值。初次生成仍因 props 默认数据未完整落地而出现 `missing_prop_defaults`，并在一轮 repair 后恢复完整 35 节点结构；因此 v6 解决的是动态属性绑定缺陷，不应把全部改进归因于初次生成已完全正确。

| 指标 | v5 | v6 | 变化 |
|---|---:|---:|---:|
| 最终结果 | 失败 | 通过 | 失败转为通过 |
| Repair 轮数 | 3 | 1 | 减少 2 轮 |
| Generation token | 21,446 | 21,123 | 减少 323 |
| Repair token | 76,285 | 25,501 | 减少 50,784 |
| 总 token | 97,731 | 46,624 | 减少 51,107（52.29%） |
| 最终 structure similarity | 0.9857 | 1.0 | +0.0143 |
| 最终 tree edit distance | 0.5 | 0 | 归零 |
| 最终关键属性错误 | 1 | 0 | 消除 |

本次结果已冻结到 `data/experiments/snapshots/pilot_9_v6_targeted_final_20261008_145944/`。快照同时保存 v5 失败基线、v6 产物、输入、配置、实际运行代码和测试，共 23 个哈希文件，SHA-256 复核差异为 0。

### 证据边界

1. 本结果是单一样本的针对性消融式对照，模型生成具有随机性且未记录固定 seed，因此可以作为机制有效的案例证据，但不能单独作为统计显著性结论。
2. `interaction_eval_passed` 为 null：当前配置没有覆盖排序、翻页、页大小切换、行选择和事件发送，现阶段结论仅覆盖静态、编译与初始状态轻量 DOM 等价。
3. 最终仍有一个 `migration_hints_not_landed` 警告。源码实际通过 `setValue()` 包装方法委托给 `this.data.set()`，属于 validator 缺少一层包装调用追踪造成的假警告，不属于当前 San 产物的业务缺陷。

下一步应先为 DataPaginationTable 增加确定性的交互场景并重评现有 v6 产物；该过程不需要重新调用模型。交互通过后，再决定是否开始 3 个 complex 样本。

In [ ]:
# 校验 v6 最终快照及 v5/v6 对照指标
v6_snapshot = project_root / 'data/experiments/snapshots/pilot_9_v6_targeted_final_20261008_145944'
assert verify_snapshot(v6_snapshot) == 23

v6_manifest = json.loads((v6_snapshot / 'manifest.json').read_text(encoding='utf-8'))
v6_result = v6_manifest['result']
comparison = v6_manifest['comparison_to_v5']

assert v6_result['final_passed'] is True
assert v6_result['repair_rounds'] == 1
assert v6_result['final_structure_similarity'] == 1.0
assert v6_result['final_tree_edit_distance'] == 0.0
assert v6_result['interaction_eval_passed'] is None
assert comparison['v5_final_passed'] is False
assert comparison['v5_repair_rounds'] == 3
assert comparison['token_reduction'] == 51107
assert comparison['token_reduction_ratio'] == 0.5229

print('v6 snapshot integrity and comparison metrics verified.')

## 17. Pilot 9 v6：DataPaginationTable 独立交互重评

在不修改冻结快照、不重新调用模型的前提下，新增独立配置 `migration_pipeline/interaction_configs/pilot_9_v6_data_pagination.json`，直接使用 v6 冻结产物，覆盖以下四个隔离场景：

| 场景 | 操作 | 关键检查 | 结果 |
|---|---|---|---|
| `sort_by_score` | 调用 `setSort('score')` | 排序字段、降序结果、当前页数据 | 通过 |
| `go_to_next_page` | 调用 `go(1)` | `page=2`、分页结果、`page` 事件 | 通过 |
| `change_page_size` | 模拟 select change，值为 6 | `pageSize=6`、`pageCount=2`、页码复位 | 通过 |
| `select_row` | 调用 `select(5)` | 选中 ID、选中行、`select` 事件 | 通过 |

首次独立运行时，Vue 与 San 的业务值、computed 结果和事件实际一致，但比较器使用 `JSON.stringify(a) === JSON.stringify(b)`，把对象键的插入顺序差异误判为场景失败。该问题属于评估器假阴性，不是迁移产物缺陷。比较器随后改为递归排序对象键后比较，同时严格保留数组顺序；专项单测及相关 25 项回归测试均通过。

最终报告为 `data/experiments/interaction_reports/pilot_9_v6_data_pagination_interactions.json`，配置指纹为 `3d7983488b7a88ba71d4c6b6a024c9a6dfb12f8e6802947fa0be51a2971afdee`。报告记录 `model_called=false`、`passed=true`、`reason=interaction_scenarios_passed`，四个场景全部通过。

### 证据边界

本次评估是在轻量、确定性的内存 Vue/San 兼容上下文中执行方法并比较状态，不是真实浏览器自动化，也不验证点击命中、CSS 选中样式、布局、截图像素或浏览器事件传播细节。它补足了 v6 的状态转换和事件语义证据，但最终论文中应将其表述为组件逻辑层交互等价性检查。

In [ ]:
# 校验 v6 DataPaginationTable 的独立交互重评报告
interaction_report_path = project_root / 'data/experiments/interaction_reports/pilot_9_v6_data_pagination_interactions.json'
interaction_report = json.loads(interaction_report_path.read_text(encoding='utf-8'))

assert interaction_report['model_called'] is False
assert interaction_report['result']['passed'] is True
assert interaction_report['result']['reason'] == 'interaction_scenarios_passed'
assert {item['name'] for item in interaction_report['result']['scenarios']} == {
    'sort_by_score', 'go_to_next_page', 'change_page_size', 'select_row'
}
assert all(item['passed'] for item in interaction_report['result']['scenarios'])
print('v6 DataPaginationTable interaction scenarios verified without a model call.')

## 18. Pilot 9 v6：首个 complex 样本 CrisisCommunicationHub

2026-10-08 正式运行 `complex_crisis_communication_hub`。初次生成未通过：静态校验报告 `missing_prop_defaults`，轻量 DOM 为 Vue 35 个节点、San 27 个节点，structure similarity 为 0.4571、tree edit distance 为 19，同时出现 San computed 求值和文本相似度警告。经过 1 轮 repair 后，`final_passed=true`、`stop_reason=passed`；最终 Vue/San 均为 35 个节点，structure/tag/text similarity 均为 1.0，tree edit distance 为 0。

本次 generation 消耗 36,308 token，repair 消耗 43,638 token，总计 79,946 token，运行约 123 秒。最终静态校验为 34/35，0 个错误、1 个警告。该 `migration_hints_not_landed` 警告仍是包装调用假阳性：最终代码通过 `setValue()` 委托给 `this.data.set()`，但 validator 只识别字段附近的直接写入。

随后使用独立配置 `migration_pipeline/interaction_configs/pilot_9_v6_crisis_communication_hub.json` 对现有产物进行无模型重评。为消除 `send()` 中 `Date.now()` 带来的非确定性，交互执行器注入 `fixed_now_ms=1791446400000`，Vue/San 两侧使用同一固定时钟。

| 场景 | 覆盖行为 | 结果 |
|---|---|---|
| `edit_subject` | 输入事件参数、草稿更新、审批重置 | 通过 |
| `toggle_audience` | 受众集合变化、接收人数重新计算 | 通过 |
| `save_version` | 新版本、活动版本、`version` 事件 | 通过 |
| `approval_workflow` | 提交审批、两级批准、`review/approve` 事件、可发送状态 | 通过 |
| `send_workflow` | 完整审批发送、发送记录、版本状态、`send` 事件 | 通过 |

交互报告 `data/experiments/interaction_reports/pilot_9_v6_crisis_communication_hub_interactions.json` 记录 `model_called=false`、`passed=true`、`reason=interaction_scenarios_passed`，配置规范化指纹为 `28fb1ec423344fbbc0cd38dbe2eea43a4746f3605635fc4c489623a1cab330f0`。相关 27 项回归测试全部通过。

最终证据冻结到 `data/experiments/snapshots/pilot_9_v6_crisis_interaction_final_20261008_155848/`。快照包含 30 个负载文件、manifest 和 SHA-256 清单；清单覆盖 31 个文件，完整性差异为 0。此前 DataPaginationTable 的冻结快照也复核为 0 差异。

### 证据边界

该样本目前具备静态/编译检查、轻量 DOM 等价和确定性内存交互等价三层自动化证据，但尚未进行真实浏览器运行、布局截图和像素比较。模型运行未记录随机 seed，因此单个 complex 成功样本不能直接外推为 complex 总体成功率。下一步应运行 `complex_warehouse_picking_wave`，继续保持一次只扩大一个样本的实验节奏。

In [ ]:
# 校验 CrisisCommunicationHub complex 快照与独立交互报告
crisis_snapshot = project_root / 'data/experiments/snapshots/pilot_9_v6_crisis_interaction_final_20261008_155848'
assert verify_snapshot(crisis_snapshot) == 31

crisis_manifest = json.loads((crisis_snapshot / 'manifest.json').read_text(encoding='utf-8'))
crisis_pipeline = crisis_manifest['pipeline_result']
crisis_interaction = crisis_manifest['interaction_evaluation']

assert crisis_pipeline['final_passed'] is True
assert crisis_pipeline['repair_rounds'] == 1
assert crisis_pipeline['final_structure_similarity'] == 1.0
assert crisis_pipeline['final_tree_edit_distance'] == 0.0
assert crisis_pipeline['total_tokens'] == 79946
assert crisis_interaction['model_called'] is False
assert crisis_interaction['passed'] is True
assert crisis_interaction['fixed_now_ms'] == 1791446400000
assert len(crisis_interaction['scenarios']) == 5
print('CrisisCommunicationHub pipeline, interaction report, and snapshot verified.')

## 19. Pilot 9 v6：第二个 complex 样本 WarehousePickingWave

2026-10-08 正式运行 `complex_warehouse_picking_wave`。初次生成因 `missing_prop_defaults` 未通过静态校验；轻量 DOM 为 Vue 30 个节点、San 18 个节点，structure similarity 为 0.1833、tree edit distance 为 24.5。经过 1 轮 repair 后，最终静态校验 35/35、0 错误、0 警告；Vue/San 均为 30 个节点，structure/tag/text similarity 均为 1.0，tree edit distance 为 0。

本次运行约 131.614 秒。generation 消耗 34,635 token，repair 消耗 41,696 token，总计 76,331 token。运行结束后，v6 checkpoint 已完成 3/9 个样本，其中 medium 1/1、complex 2/2 通过；当前累计 token 为 202,901。该统计只针对已执行样本，不是 9 样本通过率。

使用独立配置 `migration_pipeline/interaction_configs/pilot_9_v6_warehouse_picking_wave.json` 对最终产物进行确定性、无模型交互重评：

| 场景 | 关键行为 | 结果 |
|---|---|---|
| `change_picker` | change 事件参数和拣货员更新 | 通过 |
| `create_wave` | 多订单选择、行数计算、订单分配、`wave` 事件 | 通过 |
| `complete_wave` | 拣选任务完成、进度达到 100%、波次和订单完成、`complete` 事件 | 通过 |
| `exception_blocks_then_allows_completion` | 未解决异常阻止完成，解决异常后允许完成，`exception/complete` 事件 | 通过 |

异常 ID 来自 `Date.now()`，因此使用 `fixed_now_ms=1791447300000`。交互报告 `data/experiments/interaction_reports/pilot_9_v6_warehouse_picking_wave_interactions.json` 记录 `model_called=false`、`passed=true`、`reason=interaction_scenarios_passed`，配置规范化指纹为 `2c33cccdab6dd3f0de0dacc79c31a3ea893d89850d3c9bbc1e8895393b443592`。相关 28 项回归测试全部通过。

结果冻结到 `data/experiments/snapshots/pilot_9_v6_warehouse_interaction_final_20261008_160716/`。快照包含 30 个负载文件、manifest 和 SHA-256 清单；清单覆盖 31 个文件，完整性差异为 0。

### 证据边界

当前自动化证据确认了波次状态转换、computed 值和事件负载等价，但仍不是实际浏览器交互。最终 San 中 select/option 绑定、列表节点身份、选中/完成/异常颜色以及按钮点击区域仍需在真实 Vue/San runtime 中人工验收。完成该验收或明确登记风险后，再运行最后一个 complex 样本 `complex_workflow_diagram_editor`。

In [ ]:
# 校验 WarehousePickingWave complex 快照与独立交互报告
warehouse_snapshot = project_root / 'data/experiments/snapshots/pilot_9_v6_warehouse_interaction_final_20261008_160716'
assert verify_snapshot(warehouse_snapshot) == 31

warehouse_manifest = json.loads((warehouse_snapshot / 'manifest.json').read_text(encoding='utf-8'))
warehouse_pipeline = warehouse_manifest['pipeline_result']
warehouse_interaction = warehouse_manifest['interaction_evaluation']

assert warehouse_pipeline['final_passed'] is True
assert warehouse_pipeline['repair_rounds'] == 1
assert warehouse_pipeline['final_validation_warning_count'] == 0
assert warehouse_pipeline['final_structure_similarity'] == 1.0
assert warehouse_pipeline['final_tree_edit_distance'] == 0.0
assert warehouse_pipeline['total_tokens'] == 76331
assert warehouse_interaction['model_called'] is False
assert warehouse_interaction['passed'] is True
assert warehouse_interaction['fixed_now_ms'] == 1791447300000
assert len(warehouse_interaction['scenarios']) == 4
print('WarehousePickingWave pipeline, interaction report, and snapshot verified.')

## 20. WarehousePickingWave：人工浏览器发现与修复

人工打开 Vue/San 测试页并执行拣选路径点击后发现：Vue 的任务行会变为浅绿色并出现删除线，San 的任务文本虽然变为 `✓`、进度也更新，但任务行颜色没有同步变化。该问题说明静态 DOM 相似度和内存状态交互检查不足以覆盖真实浏览器的样式更新。

复核 `WarehousePickingWave_repaired.san` 后定位到两个迁移缺陷：

1. 任务行的 `class` 插值写成 `class="{{task.picked ? 'picked' : ''}"`，少了一个结束 `}`，所以 `.picked` CSS 类没有形成有效的 San 插值。
2. 多个 Vue `v-for :key` 被翻译成独立 `key="{= ... =}"`，而 San 应使用 `s-for="... trackby ..."`。任务循环未使用正确的 trackby 语义，可能导致循环节点复用时状态/class 不刷新。

修复内容：任务、订单、波次、异常和拣货员 option 循环全部改为内联 `trackby`；picker select/option 改为单向 value 插值；任务 `picked` class 插值补齐闭合分隔符。同时将这些约束加入 generation/repair Prompt，并在 Validator 中新增 `invalid_s_for_key_binding` 和 `unbalanced_template_interpolation` 错误检查。

修复后验证：San 编译通过，structure/tag/text similarity 均为 1.0，tree edit distance 为 0；Warehouse 四个确定性交互场景仍全部通过；完整回归测试 30 项通过。修复前结果保留在 `data/experiments/snapshots/pilot_9_v6_warehouse_interaction_final_20261008_160716/`，修复后单独冻结到 `data/experiments/snapshots/pilot_9_v6_warehouse_interaction_post_browser_fix_20261008_165033/`，两个快照均有 SHA-256 清单且差异为 0。

### 论文证据边界

该案例证明真实浏览器人工检查能够发现轻量 evaluator 未覆盖的模板语法和节点更新问题。因此论文中应将静态/轻量交互评估表述为第一层筛查，而将浏览器人工验证作为发现 CSS、节点复用和真实 runtime 行为缺陷的必要补充。

In [ ]:
# 校验 WarehousePickingWave 浏览器问题修复记录
post_browser_fix = project_root / 'data/experiments/snapshots/pilot_9_v6_warehouse_interaction_post_browser_fix_20261008_165033'
assert verify_snapshot(post_browser_fix) == 32
post_manifest = json.loads((post_browser_fix / 'manifest.json').read_text(encoding='utf-8'))
assert post_manifest['model_called'] is False
assert post_manifest['after_sha256'] == 'ce19a7d95d7dae8ea935517a2671b5486543fbdec632f4b9367d8a8bc9f43441'
assert post_manifest['verification']['san_compile_passed'] is True
assert post_manifest['verification']['visual_eval_passed'] is True
assert post_manifest['verification']['interaction_eval_passed'] is True
assert post_manifest['verification']['regression_tests_passed'] == 30
print('WarehousePickingWave browser-validation fix and provenance snapshot verified.')

## 21. Pilot 9 v6：第三个 complex 样本 WorkflowDiagramEditor

2026-10-12 正式运行 `complex_workflow_diagram_editor`。本次生成使用 provider `qwen`、模型 `qwen-plus`，最多 3 轮 repair。初次生成未通过静态校验，错误为 `invalid_s_for_key_binding` 和 `missing_prop_defaults`；初始视觉评估的 structure similarity 为 0.0、tag sequence similarity 为 0.68、text similarity 为 0.314、tree edit distance 为 32.25。

经过 1 轮 repair 后，`final_passed=true`、`stop_reason=passed`。最终静态校验错误数为 0，视觉评估错误数为 0；Vue/San 节点数均为 32，structure/tag/text similarity 均为 1.0，tree edit distance 为 0.0。

本次运行耗时 124.078 秒：generation 33,473 token，repair 40,780 token，总计 74,253 token。最终产物为 `data/experiments/pipeline/batches/pilot_9_v6/complex/WorkflowDiagramEditor_repaired.san`，修复历史为 `data/experiments/repair_reports/batches/pilot_9_v6/complex/WorkflowDiagramEditor_repair_history.json`。

初始视觉阶段记录了 `vue_vue_render_expression_error`、`san_san_render_expression_error` 和 `dom_text_similarity_low` warning；修复后仍保留 `migration_hints_not_landed`、`vue_vue_render_expression_error` 与 `san_san_render_expression_error` 等非阻断 warning。因此论文中应报告“最终校验通过但存在非阻断运行时表达式 warning”，不能将其描述为零诊断信息。

人工浏览器验证：将 `tests/manual/vue-test-runner.html` 加载原始 Vue，将 `tests/manual/san-test-runner.html` 切换到上述 `WorkflowDiagramEditor_repaired.san`，用户确认初始渲染、节点选择、节点操作、连线、撤销、校验和运行状态通过。该结论属于人工 runtime 验收。当前没有为 WorkflowDiagramEditor 建立独立的无模型 interaction config/report，因此 `interaction_eval_passed` 在批次报告中仍为 `null`，不能冒充自动交互评估通过。

截至本节，`pilot_9_v6` 已完成 4/9 个流水线样本：`medium_data_pagination_table`、`complex_crisis_communication_hub`、`complex_warehouse_picking_wave`、`complex_workflow_diagram_editor`；剩余 5 个样本仍为 pending。

In [ ]:
# 校验 WorkflowDiagramEditor 迁移记录与产物路径
workflow_report_path = project_root / 'data/experiments/repair_reports/batches/pilot_9_v6/complex/WorkflowDiagramEditor_repair_history.json'
workflow_output_path = project_root / 'data/experiments/pipeline/batches/pilot_9_v6/complex/WorkflowDiagramEditor_repaired.san'
workflow_report = json.loads(workflow_report_path.read_text(encoding='utf-8'))
workflow_summary = workflow_report['summary']
assert workflow_output_path.exists()
assert workflow_summary['final_passed'] is True
assert workflow_summary['repair_rounds'] == 1
assert workflow_summary['final_structure_similarity'] == 1.0
assert workflow_summary['final_tree_edit_distance'] == 0.0
assert workflow_summary['interaction_eval_passed'] is None
print('WorkflowDiagramEditor pipeline result and manual-validation provenance recorded.')